# Capitolo 5 — Supply chain con congestione e sostenibilità (LP / NLP convesso)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/laboratorio-ricerca-operativa/blob/main/notebooks/lab05_supplychain.ipynb)

Rete: 2 stabilimenti (S1, S2) → 2 hub (H1, H2) → 4 mercati (M1..M4).

Contenuto:
1. Flusso a costo minimo (LP) e prezzi ombra degli archi saturi
2. Congestione quadratica: i flussi si ripartiscono per evitare la saturazione
3. Prezzo interno della CO2 (tau): frontiera costo-emissioni
4. Minimax: minimizzare l'utilizzazione massima della rete

Il capitolo completo — modello, dati, risultati e analisi di sensitività — è [sul sito](https://fabiofurini.github.io/laboratorio-ricerca-operativa/supplychain/).

## Preparazione

La cella qui sotto installa `gurobipy` e scarica `stile.py`, la palette comune
degli script del corso. La licenza inclusa nel pacchetto pip è limitata a **2000
variabili e 2000 vincoli**: tutti i modelli del laboratorio ci stanno — il più
grande, il newsvendor a scenari, ne usa 1803 e 1801 — ma aumentando il numero di
scenari si può superarla. In quel caso si attiva la licenza accademica gratuita
da [portal.gurobi.com](https://portal.gurobi.com).

In [ ]:
# Ambiente: il solver e lo stile grafico del laboratorio.
# In locale usa il python/stile.py del repository; su Colab installa e scarica quello che manca.
import importlib.util
import subprocess
import sys
import urllib.request
from pathlib import Path

if importlib.util.find_spec("gurobipy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gurobipy", "matplotlib", "pandas", "scipy"], check=True)

if importlib.util.find_spec("stile") is None:
    locale = next((p for p in (Path("../python/stile.py"), Path("python/stile.py"))
                   if p.exists()), None)
    if locale is not None:
        sys.path.insert(0, str(locale.parent.resolve()))     # notebook aperto nel repository
    else:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/fabiofurini/laboratorio-ricerca-operativa/main/python/stile.py", "stile.py")   # Colab

In [ ]:
import gurobipy as gp
import numpy as np
import pandas as pd
from gurobipy import GRB

from stile import (ARANCIO, GRIGIO, ROSSO, TEAL, VERDE, intestazione, plt, salva_dat,
                   salva_dati, salva_figura, salva_tikz)

## 1. DATI

In [ ]:
offerta = {"S1": 260, "S2": 240}                     # capacità produttiva (unità)
domanda = {"M1": 120, "M2": 90, "M3": 140, "M4": 100}  # domanda (unità); tot 450 < 500

#           arco: (capacità U, costo unitario c €/u, emissioni e kgCO2/u)
# emissioni NON proporzionali ai costi: archi economici ma inquinanti (strada)
# e archi costosi ma puliti (ferrovia) — così il prezzo della CO2 sposta le rotte
archi = {
    ("S1", "H1"): (220, 4.0, 3.5),
    ("S1", "H2"): (180, 6.5, 1.2),
    ("S2", "H1"): (150, 7.0, 1.5),
    ("S2", "H2"): (220, 3.5, 4.0),
    ("H1", "M1"): (130, 3.0, 2.8),
    ("H1", "M2"): (100, 4.5, 1.0),
    ("H1", "M3"): (120, 5.0, 1.2),
    ("H1", "M4"): (90, 6.0, 1.0),
    ("H2", "M1"): (80, 6.0, 1.1),
    ("H2", "M2"): (90, 4.0, 2.5),
    ("H2", "M3"): (130, 3.5, 3.0),
    ("H2", "M4"): (110, 4.0, 2.4),
}
A = list(archi)
U = {a: archi[a][0] for a in A}
c = {a: archi[a][1] for a in A}
e = {a: archi[a][2] for a in A}
hub = ["H1", "H2"]

salva_dati(pd.DataFrame([(i, j, *archi[i, j]) for (i, j) in A],
                        columns=["da", "a", "capacita", "costo", "emissioni"]),
           "supplychain_archi")


def costruisci(tau=0.0, congestione=0.0):
    """Modello di flusso. tau = prezzo CO2 (€/kg); congestione = peso alpha del termine
    quadratico c_ij*x + alpha*c_ij*x^2/U (convesso)."""
    m = gp.Model("supply_chain")
    m.Params.OutputFlag = 0
    x = m.addVars(A, name="x", ub=U)
    v_off = m.addConstrs((x.sum(s, "*") <= offerta[s] for s in offerta), name="offerta")
    m.addConstrs((x.sum("*", h) == x.sum(h, "*") for h in hub), name="transito")
    v_dom = m.addConstrs((x.sum("*", k) == domanda[k] for k in domanda), name="domanda")
    obj = gp.quicksum((c[a] + tau * e[a]) * x[a] for a in A)
    if congestione > 0:
        obj += gp.quicksum(congestione * c[a] * x[a] * x[a] / U[a] for a in A)
    m.setObjective(obj, GRB.MINIMIZE)
    return m, x, v_off, v_dom


def riassunto(x):
    costo = sum(c[a] * x[a].X for a in A)
    co2 = sum(e[a] * x[a].X for a in A)
    util_max = max(x[a].X / U[a] for a in A)
    return costo, co2, util_max

## 2. LP BASE

In [ ]:
intestazione("LP: flusso a costo minimo")
m, x, v_off, v_dom = costruisci()
m.optimize()
assert m.Status == GRB.OPTIMAL
costo0, co20, util0 = riassunto(x)
print(f"Costo di trasporto: {costo0:,.2f} €   emissioni: {co20:,.1f} kgCO2   "
      f"utilizzo max archi: {util0:.0%}")
print("\nFlussi ottimi (unità) e utilizzo:")
for a in A:
    if x[a].X > 1e-6:
        print(f"  {a[0]:>2} → {a[1]:<2}: {x[a].X:6.1f} / {U[a]:3d}  ({x[a].X / U[a]:5.0%})"
              + ("   ** saturo" if x[a].X > U[a] - 1e-6 else ""))
print("\nPrezzi ombra della domanda (costo marginale di servire un'unità in più):")
for k in domanda:
    print(f"  {k}: {v_dom[k].Pi:6.2f} €/unità")
print("\nCosti ridotti degli archi non usati (di quanto deve scendere il costo unitario"
      "\ndell'arco perché entri nella soluzione ottima):")
for a in A:
    if x[a].X < 1e-6:
        print(f"  {a[0]:>2} → {a[1]:<2}: costo {c[a]:4.1f} €, RC = {x[a].RC:+5.2f} €, "
              f"range di validità SAObj = [{x[a].SAObjLow:4.1f}, +inf) "
              f"→ conveniente sotto {x[a].SAObjLow:4.1f} €/unità")
salva_dati(pd.DataFrame([(a[0], a[1], x[a].X, x[a].X / U[a]) for a in A],
                        columns=["da", "a", "flusso", "utilizzo"]), "supplychain_flussi_lp")

## 3. CONGESTIONE QUADRATICA

In [ ]:
intestazione("Congestione quadratica (alpha = 1)")
mc, xc, _, _ = costruisci(congestione=1.0)
mc.optimize()
costoc, co2c, utilc = riassunto(xc)
print(f"Costo di trasporto: {costoc:,.2f} €   emissioni: {co2c:,.1f} kgCO2   "
      f"utilizzo max archi: {utilc:.0%}")
print("Il termine quadratico ripartisce i flussi: meno archi saturi, costo lineare più alto.")

## 4. PREZZO DELLA CO2: frontiera costo-emissioni

In [ ]:
intestazione("Frontiera costo-emissioni al variare del prezzo CO2")
taus = [0, 0.5, 1, 1.5, 2, 3, 4, 6, 8, 10]
frontiera = []
for tau in taus:
    mt, xt, _, _ = costruisci(tau=tau)
    mt.optimize()
    ct, et, ut = riassunto(xt)
    frontiera.append((tau, ct, et))
    print(f"  tau = {tau:4.1f} €/kg: costo trasporto {ct:8.2f} €, emissioni {et:7.1f} kg")
front = pd.DataFrame(frontiera, columns=["tau", "costo", "emissioni"])
salva_dati(front, "supplychain_frontiera_co2")

## 5. MINIMAX: utilizzo massimo minimo

In [ ]:
intestazione("Minimax: minima utilizzazione massima della rete")
mm, xm, _, _ = costruisci()
z = mm.addVar(name="z")
mm.addConstrs((xm[a] / U[a] <= z for a in A), name="carico")
mm.setObjective(z, GRB.MINIMIZE)
mm.optimize()
print(f"Utilizzo massimo minimo possibile: {mm.ObjVal:.1%} "
      f"(LP a costo minimo: {util0:.0%}, congestione: {utilc:.0%})")

## 6. FIGURE (TikZ generato + dati pgfplots + anteprima matplotlib)

In [ ]:
pos = {"S1": (0, 1), "S2": (0, -1), "H1": (1, 0.8), "H2": (1, -0.8),
       "M1": (2, 1.5), "M2": (2, 0.5), "M3": (2, -0.5), "M4": (2, -1.5)}

salva_dat(front, "cap05_frontiera_co2")


def tikz_rete(xx, titolo):
    """Genera il codice TikZ della rete con i flussi della soluzione xx."""
    sx, sy = 3.4, 1.15                              # scala orizzontale e verticale
    r = []
    r.append(f"% Rete della soluzione: {titolo} (generato da lab05_supplychain.py)")
    r.append("\\begin{tikzpicture}[>=stealth,")
    r.append("    nodo/.style={circle, draw=none, text=white, font=\\bfseries\\small,")
    r.append("                 minimum size=8mm, inner sep=0pt}]")
    for a in A:
        (x1, y1), (x2, y2) = pos[a[0]], pos[a[1]]
        f = xx[a].X
        if f > 1e-6:
            saturo = f > U[a] - 1e-6
            colore = "rossomattone" if saturo else "teal"
            spess = 0.4 + 1.6 * f / max(U.values())
            r.append(f"  \\draw[{colore}, line width={spess:.2f}pt] "
                     f"({x1 * sx:.2f},{y1 * sy:.2f}) -- ({x2 * sx:.2f},{y2 * sy:.2f})")
            r.append(f"    node[midway, above, sloped, font=\\tiny, text=black!60] "
                     f"{{{f:.0f}}};")
        else:
            r.append(f"  \\draw[black!25, densely dotted, line width=0.4pt] "
                     f"({x1 * sx:.2f},{y1 * sy:.2f}) -- ({x2 * sx:.2f},{y2 * sy:.2f});")
    stile_nodo = {"S": "verde", "H": "arancio", "M": "teal"}
    for nn, (px, py) in pos.items():
        r.append(f"  \\node[nodo, fill={stile_nodo[nn[0]]}] at ({px * sx:.2f},{py * sy:.2f}) "
                 f"{{{nn}}};")
    r.append(f"  \\node[font=\\small\\bfseries, text=blunotte] at ({sx:.2f},{2.0 * sy:.2f}) "
             f"{{{titolo}}};")
    r.append("\\end{tikzpicture}")
    return "\n".join(r)


salva_tikz(tikz_rete(x, "LP a costo minimo"), "cap05_rete_lp")
salva_tikz(tikz_rete(xc, "Congestione quadratica"), "cap05_rete_congestione")

fig, assi = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (xx, titolo) in zip(assi, [(x, "LP costo minimo"), (xc, "Congestione quadratica")]):
    for a in A:
        (x1, y1), (x2, y2) = pos[a[0]], pos[a[1]]
        f = xx[a].X
        if f > 1e-6:
            colore = ROSSO if f > U[a] - 1e-6 else TEAL
            ax.plot([x1, x2], [y1, y2], color=colore, lw=0.6 + 4.5 * f / max(U.values()),
                    alpha=0.85, zorder=1)
            ax.annotate(f"{f:.0f}", ((x1 + x2) / 2, (y1 + y2) / 2 + 0.06),
                        fontsize=7, color=GRIGIO, ha="center")
        else:
            ax.plot([x1, x2], [y1, y2], color=GRIGIO, lw=0.5, ls=":", alpha=0.4, zorder=0)
    for n, (px, py) in pos.items():
        col = VERDE if n.startswith("S") else (ARANCIO if n.startswith("H") else TEAL)
        ax.scatter([px], [py], s=520, color=col, zorder=2)
        ax.annotate(n, (px, py), color="white", weight="bold", ha="center", va="center", zorder=3)
    ax.set_title(titolo + " (rosso = arco saturo)")
    ax.axis("off")
salva_figura(fig, "cap05_reti")

fig, ax = plt.subplots()
ax.plot(front["emissioni"], front["costo"], "-o", color=TEAL)
for _, r in front.iterrows():
    if r["tau"] in (0, 1, 2, 4, 10):
        ax.annotate(f"  $\\tau$={r['tau']:.0f}", (r["emissioni"], r["costo"]), fontsize=8)
ax.set_xlabel("emissioni totali (kgCO$_2$)")
ax.set_ylabel("costo di trasporto (€)")
ax.set_title("Frontiera costo-emissioni al crescere del prezzo interno della CO$_2$")
salva_figura(fig, "cap05_frontiera_co2")

print("\nFatto: capitolo 5.")

---

Notebook generato da `python/lab05_supplychain.py` con `python3 python/genera_notebook.py`:
le modifiche si fanno sullo script, non qui.

Materiale didattico di [Fabio Furini](https://sites.google.com/view/fabiofurini/home-page) — DIAG, Sapienza Università di Roma.
Testi, figure e dati [CC BY 4.0](https://github.com/fabiofurini/laboratorio-ricerca-operativa/blob/main/LICENSE),
codice [MIT](https://github.com/fabiofurini/laboratorio-ricerca-operativa/blob/main/LICENSE-CODE).